# Train MelaDx7 on HAM10000 (Kaggle or Colab GPU)

This notebook runs the repository's own training pipeline on a free GPU and produces a
model artifact you can drop into `models/` on your machine.

**Before running**

1. Enable a **GPU** accelerator and **Internet** in the notebook settings.
2. Add data. Either:
   * **A. Prepared split (used for the release model):** attach the uploaded Kaggle dataset
     containing `lesionlens-code` (zip of this repository) and `lesionlens-ham10000-processed`
     (the output of `python -m ml.datasets.prepare_ham10000`, already lesion-grouped), or
   * **B. Raw HAM10000:** attach "Skin Cancer MNIST: HAM10000" (kmader) and set `REPO_URL`.
3. Pick the model with `CHOICE` in the first code cell: `vit` (ViT-Base/16), `b0` or `b3`.
4. HAM10000 is **CC BY-NC 4.0** (non-commercial, attribution to Tschandl et al.,
   *Sci. Data* 5, 180161, 2018).

Nothing here fabricates data or results: metrics come from the held-out, lesion-grouped
test split and are written to `metrics.json` next to the weights.


In [ ]:
import glob, os, shutil, subprocess
# --- choose the model -------------------------------------------------------------
CHOICE = "vit"   # "vit" | "b0" | "b3"
# vit: ViT-Base/16 @ 224 px, fine-tuned from Google's public ImageNet-21k weights
# b0 : EfficientNet-B0 @ 224 px (the v1.0.0 CNN)      b3: EfficientNet-B3 @ 300 px
CFG = {
    "vit": ("ml/configs/vit_base.yaml", "vit_base_patch16_224-v1.0.0", 32, ""),
    "b0": ("ml/configs/efficientnet_b0.yaml", "efficientnet_b0-v1.0.0", 64, ""),
    "b3": ("ml/configs/efficientnet_b0.yaml", "efficientnet_b3-v1.0.0", 32,
           "--set model.architecture=efficientnet_b3 --set model.input_size=300 "
           "--set experiment_name=efficientnet_b3_ham10000 --set training.epochs=25"),
}
CONFIG, MODEL_NAME, BATCH, EXTRA = CFG[CHOICE]
# ----------------------------------------------------------------------------------
MODE = "prepared"   # "prepared" (option A) or "raw" (option B)
REPO_URL = "https://github.com/<your-account>/lesionlens.git"  # option B only
DATASET_DIR = "/kaggle/input/skin-cancer-mnist-ham10000"       # option B only

if MODE == "prepared":
    code = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    assert code, "attach the dataset containing lesionlens-code"
    shutil.copytree(os.path.dirname(code[0]), "/kaggle/working/lesionlens", dirs_exist_ok=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/lesionlens"], check=True)
%cd /kaggle/working/lesionlens
!pip install -q -r ml/requirements.txt && pip install -q -e .


In [ ]:
import torch, os, glob
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if MODE == "raw":
    print(sorted(os.listdir(DATASET_DIR)))
    image_dirs = sorted(d for d in glob.glob(f"{DATASET_DIR}/*") if os.path.isdir(d) and "images" in d.lower())
    print(image_dirs)


## 1. Lesion-grouped split

Several HAM10000 images show the same lesion. Splitting by `lesion_id` keeps every image
of a lesion in one split, so test metrics are not inflated by near-duplicates.

In [ ]:
if MODE == "prepared":
    manifest = glob.glob("/kaggle/input/**/split_manifest.csv", recursive=True)
    if not manifest:
        print("Inputs visible to this notebook:")
        for d in sorted(os.listdir("/kaggle/input")):
            print(" ", d, sorted(os.listdir(f"/kaggle/input/{d}"))[:10])
    assert manifest, "split_manifest.csv not found: the dataset with lesionlens-ham10000-processed is not attached (or its files are missing)"
    root = os.path.dirname(manifest[0])
    os.makedirs("data", exist_ok=True)
    if os.path.lexists("data/processed"): os.remove("data/processed")
    os.symlink(root, "data/processed")     # read-only inputs are fine: training only reads
    print(open(f"{root}/split_summary.json").read()[:1500])
else:
    dirs = " ".join(image_dirs)
    !python -m ml.datasets.prepare_ham10000 --metadata {DATASET_DIR}/HAM10000_metadata.csv --images {dirs} --output data/processed --mode symlink


## 2. Train

EfficientNet-B0 from ImageNet weights, class-weighted loss, warm-up + cosine schedule,
early stopping on validation macro-F1, temperature calibration, then evaluation on the
held-out test split. Override any setting with `--set key=value`.

In [ ]:
!python -m ml.training.train --config {CONFIG} --set data.num_workers=2 --set data.batch_size={BATCH} {EXTRA}


In [ ]:
import json
m = json.load(open(f"models/{MODEL_NAME}/metrics.json"))["metrics"]
print({k: m[k] for k in ("n_samples", "accuracy", "balanced_accuracy", "roc_auc_macro_ovr")}, "macro-F1:", m["macro"]["f1"])
for row in m["per_class"]:
    print(f"{row['code']:>6}  support={row['support']:>4}  recall={row['recall']}  specificity={row['specificity']}  auc={row['roc_auc']}")

## 3. Download the artifact

Unzip it into `models/` in your local repository and set
`MODEL_PATH=models/efficientnet_b0-v1.0.0` (or `MODEL_DIR=efficientnet_b0-v1.0.0` for Docker).
The `checkpoints/` folder is only needed to resume training and is excluded.

In [ ]:
!cd models && zip -qr ../{MODEL_NAME}.zip {MODEL_NAME} -x "*/checkpoints/*" && cd .. && ls -lh {MODEL_NAME}.zip